# SPARQL — User Guide

Querying RDF 1.2 data: triple-term-aware functions, quoted-triple patterns, and annotation shorthand in queries.

See the [Graphs guide](02-graphs.ipynb) for the `TripleTerm`/`DirLangString`/reification semantics the query examples below build on, and [SHACL rules](04b-shacl-inference-rules.ipynb) for `sh:sparql`/`sh:SPARQLRule`, which use these same query functions inside SHACL shapes.

Two related guides worth knowing about: **[3.a SPARQL rules (pending)](03a-sparql-rules-pending.md)** — SPARQL-RL (SRL), a separate Datalog-style rules language, deliberately out of scope for this project. **[5.c SPARQL queries as RDF](05c-sparql-query-as-rdf.ipynb)** — treating a *query itself* as RDF data you can encode, inspect, edit, and validate, a separate concern from writing or running one.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells from top to bottom — later sections reuse variables from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## Query semantics and built-in functions

Supports SPARQL 1.2 query execution:
- Query over reified quoted triples
- Turtle 1.2 quoted-triple syntax in queries (`<<( ... )>>`)
- Binds variables for terms inside triple terms (`?s ?p ?o`)

The examples below run against one running example graph, parsed here (the same document used in the [serialization formats guide](05a-serialization-formats.ipynb)'s `turtle12` example).

In [2]:
g_parsed = StarLayerGraph()
g_parsed.bind("ex", EX)
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals
    ex:note_en ex:text "hello"@en--ltr .
    ex:note_ar ex:text "مرحبا"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:source ex:wikipedia ;
      ex:confidence "high" .

    # anonymous inline annotation block
    ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} .

    # named reifier with annotations
    ex:bob ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} .

    # named reifier without annotation block
    ex:bob ex:worksWith ex:frank ~ ex:stmt2 .

    # an additional quoted triple term reused in the query examples below
    ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .
''', format='turtle12')
print("parsed triples:", len(g_parsed))

parsed triples: 16


In [3]:
# SPARQL query to bind all terms inside a quoted triple
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:knows)
}
ORDER BY ?claim ?s ?o
""")

for row in rows:
    print(
        g_parsed.qname(row.claim),
        g_parsed.qname(row.s),
        g_parsed.qname(row.p),
        g_parsed.qname(row.o),
        "Source: ", g_parsed.qname(row.source),
    )

ex:claim ex:bob ex:knows ex:carol Source:  ex:wikipedia


In [4]:
# detect triple-term values using isTRIPLE
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?p WHERE {
  ?claim ?p ?statement .
  FILTER( isTRIPLE(?statement) )
}
ORDER BY ?claim
""")

for row in rows:
    print(g_parsed.qname(row.claim), g_parsed.qname(row.p))

ex:alice ex:mentions
ex:claim rdf:reifies
ex:stmt1 rdf:reifies
ex:stmt2 rdf:reifies
rr:0 rdf:reifies


### Additional SPARQL 1.2 functions

- `TRIPLE(s, p, o)` — a functional way to write a triple term `<<( s p o )>>`
- `SUBJECT()`, `PREDICATE()`, `OBJECT()` — extract the parts out of a triple term
- `LANGDIR()`, `hasLANGDIR()`, `STRLANGDIR()` — read, check, and build a literal's base direction
- `LANG()`, `hasLANG()` — SPARQL 1.1 functions, now also aware of direction-tagged literals

In [5]:
# TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:bob, ex:knows, ex:carol))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname(row.s), g_parsed.qname(row.p), g_parsed.qname(row.o))

ex:bob ex:knows ex:carol


In [6]:
# LANGDIR() / hasLANGDIR() / LANG() / hasLANG() over the direction-tagged notes
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?s ?lang ?dir ?hasDir WHERE {
  ?s ex:text ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s
""")
for row in rows:
    print(g_parsed.qname(row.s), row.lang, row.dir, row.hasDir)

# STRLANGDIR() constructs a direction-tagged literal directly from plain strings
rows = g_parsed.query('SELECT ?lit WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }')
for row in rows:
    print(row.lit.n3())

ex:note_ar ar rtl true
ex:note_en en ltr true
"hi"@en--ltr


### Turtle annotation shorthand inside SPARQL queries

The `{| ?pred ?val |}`, `~ ?r`, and `<< s p o >>` Turtle syntax used to parse and serialize graphs also works directly inside a SPARQL 1.2 `WHERE` clause — querying using the same shorthand a graph is serialized with, without expanding to `rdf:reifies`/`<<( )>>` by hand.

In [7]:
# {| ?pred ?val |}: query an anonymous reifier's annotations inline
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  ex:bob ex:likes ex:dana {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

# ~ ?r: bind the reifier itself for a named reifier
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?r WHERE {
  ex:bob ex:worksWith ex:frank ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname(row.r))

# << s p o >> ?pred ?val: reification shorthand, no assertion required
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  << ex:bob ex:likes ex:dana >> ?pred ?val
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

ex:since 2020
ex:source http://example.org/LinkedIn
ex:stmt2
ex:since 2020
ex:source http://example.org/LinkedIn


## Further work

- **SPARQL-RL (SRL) rules** are deliberately out of scope for this project — see [3.a](03a-sparql-rules-pending.md).
- **Treating a query as RDF data** — encoding, editing, and validating a query itself, a separate concern from writing or running one — is covered in [5.c](05c-sparql-query-as-rdf.ipynb).
- **`sh:sparql` and `sh:SPARQLRule`** use these same query functions from inside SHACL shapes — see the [SHACL inference rules guide](04b-shacl-inference-rules.ipynb).